# P1 Rate Limiter Test — Pipeline Validation

**Mục tiêu:** Xác nhận `rate_limited_llm_call()` loại bỏ lỗi 429 trong pipeline.

**Test plan:**
1. Unit test: Rate limiter module (bucket, circuit breaker)
2. Smoke 5Q: Verify pipeline still works with rate limiter wrapper
3. **Large 30Q: Key test — should eliminate 429 errors (baseline: 64.1% pass rate)**
4. Summary metrics

**Baseline (M3 without rate limiter):**
| Test | Pass% | Time |
|------|-------|------|
| smoke_5q | 100% | 64.2s |
| large_30q | 64.1% | 125.0s |

In [1]:
# Cell 1: Setup & Imports
import sys, json, time, asyncio
from pathlib import Path

project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import get_settings
from src.api.schemas import (
    GenerationRequest, GameType, DifficultyLevel, DocScope,
    GameContentResponse, QuizQuestion, Flashcard, FillBlankQuestion,
)
from src.graph.builder import compile_graph
from src.graph.state import MAX_REVIEW_ITERATIONS

settings = get_settings()
print(f"Project: {settings.gcp_project_id}")
print(f"Generation model: {settings.generation_model} @ {settings.generation_model_location or settings.gcp_location}")
print(f"Review model: {settings.review_model} @ {settings.review_model_location}")
print(f"Rate limit: {settings.llm_rate_limit_rpm} RPM, max concurrent: {settings.llm_max_concurrent}")
print(f"Max review iterations: {MAX_REVIEW_ITERATIONS}")

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: green-mercury-485016-n1
Generation model: gemini-2.5-flash @ asia-southeast1
Review model: gemini-3.1-flash-lite-preview @ global
Rate limit: 60 RPM, max concurrent: 10
Max review iterations: 3


In [2]:
# Cell 2: Helpers
def pp(data):
    if hasattr(data, "model_dump"):
        data = data.model_dump()
    print(json.dumps(data, ensure_ascii=False, indent=2, default=str))

def make_request(
    topic: str,
    num_questions: int = 10,
    game_types: list[str] | None = None,
    difficulty: str = "comprehension",
    doc_scope: str = "system",
) -> GenerationRequest:
    gts = [GameType(g) for g in (game_types or ["quiz"])]
    return GenerationRequest(
        user_id="rate_limiter_test",
        topic=topic,
        game_types=gts,
        num_questions=num_questions,
        difficulty=DifficultyLevel(difficulty),
        doc_scope=DocScope(doc_scope),
    )

async def run_pipeline(request: GenerationRequest) -> dict:
    """Run pipeline with fresh compile (no singleton cache)."""
    app = compile_graph()
    initial_state = {
        "request": request,
        "doc_scope": request.doc_scope.value,
        "iteration_count": 0,
        "rejected_items": [],
        "errors": [],
    }
    t0 = time.time()
    result = await app.ainvoke(initial_state)
    elapsed = time.time() - t0
    return {**result, "_elapsed": elapsed}

def summarize(result: dict, label: str = "") -> dict:
    """Summarize pipeline result."""
    output = result.get("final_output")
    elapsed = result.get("_elapsed", 0)
    errors = result.get("errors", [])
    iterations = result.get("iteration_count", 0)

    quiz_count = len(output.content.quiz) if output else 0
    fc_count = len(output.content.flashcard) if output else 0
    fb_count = len(output.content.fill_blank) if output else 0
    total = quiz_count + fc_count + fb_count

    generated = output.metadata.total_generated if output else 0
    passed = output.metadata.total_passed_review if output else 0
    rejected = output.metadata.total_rejected if output else 0

    # Check for 429 errors
    error_429 = [e for e in errors if "429" in str(e) or "resource exhausted" in str(e).lower()]

    status = "✅" if total > 0 and not error_429 else "❌"
    print(f"{status} [{label}] {total} items in {elapsed:.1f}s ({iterations} iter)")
    print(f"   Quiz: {quiz_count}, FC: {fc_count}, FB: {fb_count}")
    print(f"   Generated: {generated}, Passed: {passed}, Rejected: {rejected}")
    if error_429:
        print(f"   ⚠️  429 ERRORS: {len(error_429)}")
    if errors:
        print(f"   Errors ({len(errors)}): {[str(e)[:80] for e in errors[:3]]}")

    return {
        "label": label, "total": total, "quiz": quiz_count, "fc": fc_count, "fb": fb_count,
        "generated": generated, "passed": passed, "rejected": rejected,
        "elapsed": elapsed, "iterations": iterations,
        "errors_429": len(error_429), "errors_total": len(errors),
    }

print("Helpers ready ✓")

Helpers ready ✓


## Unit Test: Rate Limiter Module

In [3]:
# Cell 3: Unit test — AsyncTokenBucket rate limiting behavior
from src.services.rate_limiter import (
    AsyncTokenBucket, CircuitBreaker, rate_limited_llm_call, CircuitBreakerOpenError,
)

# Test 1: Token bucket consumes tokens correctly
bucket = AsyncTokenBucket(capacity=3.0, refill_rate=1.0)
t0 = time.time()

async def test_bucket():
    # Should consume 3 tokens instantly (burst)
    for i in range(3):
        await bucket.wait_and_consume()
    burst_time = time.time() - t0
    print(f"  3 burst tokens consumed in {burst_time:.3f}s (should be < 0.1s)")
    assert burst_time < 0.5, f"Burst too slow: {burst_time}s"

    # 4th token should wait ~1s for refill
    t1 = time.time()
    await bucket.wait_and_consume()
    wait_time = time.time() - t1
    print(f"  4th token waited {wait_time:.3f}s (should be ~1s)")
    assert wait_time > 0.5, f"Should have waited: {wait_time}s"
    print("✅ Token bucket: burst + rate limiting works")

await test_bucket()

# Test 2: Circuit breaker trips and recovers
cb = CircuitBreaker(failure_threshold=3, cooldown_seconds=1.0)
assert not cb.is_open, "Should start closed"
cb.record_failure()
cb.record_failure()
cb.record_failure()
assert cb.is_open, "Should be open after 3 failures"
print("✅ Circuit breaker: trips after threshold")

# Wait for cooldown
await asyncio.sleep(1.1)
assert not cb.is_open, "Should be half-open after cooldown"
cb.record_success()
assert not cb.is_open, "Should be closed after success"
print("✅ Circuit breaker: recovers after cooldown + success")

# Test 3: rate_limited_llm_call wraps a coroutine
async def mock_llm_call():
    return {"content": "test response"}

result = await rate_limited_llm_call(mock_llm_call())
assert result == {"content": "test response"}
print("✅ rate_limited_llm_call: wraps coroutine correctly")

print("\n🎯 All unit tests passed!")

  3 burst tokens consumed in 0.000s (should be < 0.1s)
  4th token waited 1.001s (should be ~1s)
✅ Token bucket: burst + rate limiting works
2026-03-26 15:17:52 [warning  ] circuit_breaker_tripped        consecutive_failures=3 cooldown_seconds=1.0
✅ Circuit breaker: trips after threshold
2026-03-26 15:17:53 [info     ] circuit_breaker_half_open      msg='allowing retry after cooldown'
✅ Circuit breaker: recovers after cooldown + success
2026-03-26 15:17:53 [info     ] rate_limiter_init_bucket       capacity=5.0 refill_rate=1.0 rpm=60
2026-03-26 15:17:53 [info     ] rate_limiter_init_semaphore    max_concurrent=10
✅ rate_limited_llm_call: wraps coroutine correctly

🎯 All unit tests passed!


## Integration Test 1: Smoke 5Q (baseline sanity check)

In [4]:
# Cell 4: Smoke 5Q — verify pipeline works with rate limiter
results = []

req = make_request("Hàm số bậc hai — Toán 10", num_questions=5)
result = await run_pipeline(req)
s = summarize(result, "smoke_5q")
results.append(s)

assert s["total"] >= 3, f"Expected ≥3 items, got {s['total']}"
assert s["errors_429"] == 0, f"Got {s['errors_429']} 429 errors!"
print(f"\n✅ Smoke 5Q passed — {s['total']} items, 0 rate limit errors")

2026-03-26 15:18:08 [info     ] creating_graph                
2026-03-26 15:18:08 [info     ] graph_created                 
2026-03-26 15:18:08 [info     ] graph_compiled_without_checkpointer
2026-03-26 15:18:08 [info     ] supervisor_analyzing           doc_scope=system topic='Hàm số bậc hai — Toán 10' user_id=rate_limiter_test
2026-03-26 15:18:08 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


/home/sakana/Code/ManifoldsTeam/AIServices/src/services/llm.py:99: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  return ChatVertexAI(


2026-03-26 15:18:11 [info     ] supervisor_classified          content_type=math
2026-03-26 15:18:11 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-26 15:18:11 [info     ] math_agent_starting            doc_scope=system num_questions=5 num_to_generate=7 topic='Hàm số bậc hai — Toán 10' user_id=rate_limiter_test
2026-03-26 15:18:11 [info     ] retrieving_context             doc_scope=system query='Hàm số bậc hai — Toán 10 comprehension level educational content' user_id=rate_limiter_test
2026-03-26 15:18:11 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=rate_limiter_test
2026-03-26 15:18:11 [debug    ] retrieving_documents           doc_scope=system query='Hàm số bậc hai — Toán 10 comprehension level educational content'


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-26 15:18:13 [debug    ] retrieved_documents            count=30
2026-03-26 15:18:13 [info     ] context_retrieved              chunks=30 sources=30
2026-03-26 15:18:13 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-26 15:18:13 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-26 15:18:54 [info     ] math_agent_code_execution_done code_traces_count=12 generation_attempt=1 text_length=8866
2026-03-26 15:18:54 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-26 15:19:19 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=7
2026-03-26 15:19:19 [info     ] math_agent_generated           items_count=7
2026-03-26 15:19:19 [info     ] reviewer_reviewing             items_count=7 iteration=0
2026-03-26 15:19:19 [debug    ] creating_review_llm            location=global max_

## Integration Test 2: Large 30Q (KEY TEST — should eliminate 429 errors)

**Baseline WITHOUT rate limiter:** 64.1% pass rate, 429 errors
**Target WITH rate limiter:** ≥90% pass rate, 0 rate limit errors

In [5]:
# Cell 5: Large 30Q — THE key test for rate limiter effectiveness
req = make_request("Phương trình và bất phương trình — Toán 10", num_questions=30)
result = await run_pipeline(req)
s = summarize(result, "large_30q")
results.append(s)

pass_rate = (s["passed"] / s["generated"] * 100) if s["generated"] > 0 else 0
print(f"\n--- Large 30Q Results ---")
print(f"Pass rate: {pass_rate:.1f}% (baseline: 64.1%)")
print(f"429 errors: {s['errors_429']} (baseline: multiple)")
print(f"Time: {s['elapsed']:.1f}s (baseline: 125.0s)")

if s["errors_429"] == 0:
    print("✅ Rate limiter working — no 429 errors!")
else:
    print("❌ Still getting 429 errors — needs investigation")

2026-03-26 15:19:52 [info     ] creating_graph                
2026-03-26 15:19:52 [info     ] graph_created                 
2026-03-26 15:19:52 [info     ] graph_compiled_without_checkpointer
2026-03-26 15:19:52 [info     ] supervisor_analyzing           doc_scope=system topic='Phương trình và bất phương trình — Toán 10' user_id=rate_limiter_test
2026-03-26 15:19:52 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0
2026-03-26 15:19:53 [info     ] supervisor_classified          content_type=math
2026-03-26 15:19:53 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-26 15:19:53 [info     ] math_agent_starting            doc_scope=system num_questions=30 num_to_generate=39 topic='Phương trình và bất phương trình — Toán 10' user_id=rate_limiter_test
2026-03-26 15:19:53 [info     ] retrieving_context             doc_scope=system query='Phương trình và bất phương trình — Toán 

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-26 15:19:55 [debug    ] retrieved_documents            count=30
2026-03-26 15:19:55 [info     ] context_retrieved              chunks=30 sources=30
2026-03-26 15:19:55 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-26 15:19:55 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-26 15:22:29 [info     ] math_agent_code_execution_done code_traces_count=26 generation_attempt=1 text_length=27473
2026-03-26 15:22:29 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-26 15:22:37 [info     ] math_agent_batch_parsed        attempt=1 batch=4 items=7
2026-03-26 15:22:37 [info     ] math_agent_batch_parsed        attempt=1 batch=6 items=4
2026-03-26 15:22:38 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=7
2026-03-26 15:22:40 [info     ] math_agent_batch_parsed        attempt=1 

## Integration Test 3: Multi-type 15Q (quiz + flashcard + fill_blank)

In [6]:
# Cell 6: Multi-type — stresses rate limiter with parallel formatter calls
req = make_request(
    "Vectơ và hình học không gian — Toán 11",
    num_questions=15,
    game_types=["quiz", "flashcard", "fill_blank"],
)
result = await run_pipeline(req)
s = summarize(result, "multi_type_15q")
results.append(s)

assert s["errors_429"] == 0, f"Got {s['errors_429']} 429 errors!"
print(f"\n✅ Multi-type 15Q passed — {s['total']} items across 3 game types")

2026-03-26 15:24:13 [info     ] creating_graph                
2026-03-26 15:24:13 [info     ] graph_created                 
2026-03-26 15:24:13 [info     ] graph_compiled_without_checkpointer
2026-03-26 15:24:13 [info     ] supervisor_analyzing           doc_scope=system topic='Vectơ và hình học không gian — Toán 11' user_id=rate_limiter_test
2026-03-26 15:24:13 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0
2026-03-26 15:24:16 [info     ] supervisor_classified          content_type=math
2026-03-26 15:24:16 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-26 15:24:16 [info     ] math_agent_starting            doc_scope=system num_questions=15 num_to_generate=20 topic='Vectơ và hình học không gian — Toán 11' user_id=rate_limiter_test
2026-03-26 15:24:16 [info     ] retrieving_context             doc_scope=system query='Vectơ và hình học không gian — Toán 11 comprehen

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-26 15:24:18 [debug    ] retrieved_documents            count=30
2026-03-26 15:24:18 [info     ] context_retrieved              chunks=30 sources=30
2026-03-26 15:24:18 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-26 15:24:18 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-26 15:28:14 [info     ] math_agent_code_execution_done code_traces_count=40 generation_attempt=1 text_length=10058
2026-03-26 15:28:14 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-26 15:28:19 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-26 15:28:27 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=6
2026-03-26 15:28:35 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-26 15:28:35 [info     ] math_agent_parse_succeeded     generation

## Summary

In [7]:
# Cell 7: Summary table
print("=" * 80)
print("P1 RATE LIMITER TEST SUMMARY")
print("=" * 80)
print(f"{'Test':<20} {'Total':>6} {'Quiz':>5} {'FC':>5} {'FB':>5} {'Pass%':>7} {'429s':>5} {'Time':>8}")
print("-" * 80)

total_items = 0
total_429 = 0
total_time = 0

for r in results:
    pass_pct = (r["passed"] / r["generated"] * 100) if r["generated"] > 0 else 0
    print(
        f"{r['label']:<20} {r['total']:>6} {r['quiz']:>5} {r['fc']:>5} {r['fb']:>5}"
        f" {pass_pct:>6.1f}% {r['errors_429']:>5} {r['elapsed']:>7.1f}s"
    )
    total_items += r["total"]
    total_429 += r["errors_429"]
    total_time += r["elapsed"]

print("-" * 80)
print(f"{'TOTAL':<20} {total_items:>6} {'':>5} {'':>5} {'':>5} {'':>7} {total_429:>5} {total_time:>7.1f}s")
print()

if total_429 == 0:
    print("🎯 P1 RATE LIMITER: SUCCESS — Zero 429 errors across all tests!")
else:
    print(f"⚠️  P1 RATE LIMITER: {total_429} 429 errors remain — needs tuning")

P1 RATE LIMITER TEST SUMMARY
Test                  Total  Quiz    FC    FB   Pass%  429s     Time
--------------------------------------------------------------------------------
smoke_5q                  5     5     0     0  100.0%     0    88.6s
large_30q                30    30     0     0  100.0%     0   244.3s
multi_type_15q           45    15    15    15  100.0%     0   302.3s
--------------------------------------------------------------------------------
TOTAL                    80                               0   635.2s

🎯 P1 RATE LIMITER: SUCCESS — Zero 429 errors across all tests!
